Notebooks to analyse the survey conducted between December 2025-April 2026 within the interdisciplinary AHOI Project.

Use this notebook for the evaluation of questionnaires (Likert scale).
Use the "*Analysis_Text* notebook for the evaluation of open questions.

The survey outline and proposed research has been published here:
https://link.springer.com/chapter/10.1007/978-3-032-19099-4_8

In [ ]:
# Import first necessary libraries for data handling, statistics, and plotting.
# Each cell contains also individual libs to comment in if only specific part is needed.
# Some cells also contain optional print commands to check if you are actually getting the correct data and dimensions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Change file name here, if necessary
df = pd.read_csv('/content/AHOI_Survey_HumanAITeaming_FINISHED_TRUE_values.csv')

# Optional print cmds to check dimensions and first db entries
#print(f"Original DataFrame shape: {df.shape}")
#print("Original DataFrame head:")
#display(df.head())#
#print(df.columns)

Evaluation ATAS: Abbreviated Technology Anxiety Scale. Used as a pre-questionnaire.

Extract only the ATAS responses (5-point Likert scale); includes data cleaning (e.g. removal of strings, NaN removal, etc.)

In [ ]:
# Select only the 'ATAS' columns (pre-questionnaire)

atas_columns = [col for col in df.columns if 'ATAS' in col]
df_atas = df[atas_columns].copy()

# Optional print for dimension check and first db entries
#print(f"DataFrame with ATAS columns shape: {df_atas.shape}")
#print("ATAS DataFrame head:")
#display(df_atas.head())

# Convert all ATAS columns to numeric, coercing errors to NaN
for col in df_atas.columns:
    df_atas[col] = pd.to_numeric(df_atas[col], errors='coerce')

# Remove rows where all ATAS columns are NaN (these rows likely contained only text)
df_atas_cleaned = df_atas.dropna(how='all')

# Keep only values between 1 and 5 (inclusive)
df_atas_cleaned = df_atas_cleaned[(df_atas_cleaned >= 1).all(axis=1) & (df_atas_cleaned <= 5).all(axis=1)]

# Reset index
df_atas_cleaned = df_atas_cleaned.reset_index(drop=True)

# Save the new dataframe to a new csv file
output_file_name = 'ATAS.csv'
df_atas_cleaned.to_csv(output_file_name, index=False)

# Optional print for dimension check and first db entries
#print(f"Cleaned ATAS DataFrame shape: {df_atas_cleaned.shape}")
#print("Cleaned ATAS DataFrame head:")
display(df_atas_cleaned.head())
#print(f"Cleaned data saved to '{output_file_name}'")

Calculate desired (descriptive) statistics.

In [ ]:
# Calculate (descriptive) statistics
stats = df_atas_cleaned.describe().T
stats['median'] = df_atas_cleaned.median()
stats = stats[['mean', 'median', 'std', 'min', 'max']]

print("Descriptive Statistics for ATAS Likert Items:")
display(stats)

Plot results.

In [ ]:
# Plot response distribution (good for first check but difficult to read).
# Comment in the libraries if only plotting needed
#import matplotlib.pyplot as plt
#import seaborn as sns

plt.figure(figsize=(12, 6))
df_melted = df_atas_cleaned.melt(var_name='Item', value_name='Score')

sns.countplot(data=df_melted, x='Item', hue='Score', palette='viridis')
plt.title('Distribution of Likert Scale Responses per ATAS Item')
plt.xlabel('ATAS Question')
plt.ylabel('Frequency')
plt.legend(title='Score (1-5)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Creates bar chart per item with median and std (mean just for reference as Likert is ordinal data)
# Comment-in import command if used as stand-alone
#import matplotlib.pyplot as plt
#import seaborn as sns
#import pandas as pd
#import numpy as np

# Calculate means, standard deviations, and medians
means = df_atas_cleaned.mean()
stds = df_atas_cleaned.std()
medians = df_atas_cleaned.median()

plt.figure(figsize=(10, 6))

# Create the bar plot with whiskers indicating the standard deviation (sd)
sns.barplot(data=df_atas_cleaned, palette='viridis', errorbar='sd', capsize=0.1)

# Add median and mean values for reference
plt.scatter(x=range(len(medians)), y=medians, color='blue', zorder=5, label='Median', marker='o')
plt.scatter(x=range(len(means)), y=means, color='red', marker='*', s=100, zorder=6, label='Mean')

plt.title('Mean and Median ATAS Scores with SD Whiskers')
plt.xlabel('ATAS Items')
plt.ylabel('Likert Scale (1-5)')
plt.ylim(1, 5)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

Response distribution for each ATAS item individually to see where the spread is coming from.

In [ ]:
#import matplotlib.pyplot as plt
#import seaborn as sns
import math

num_cols = 3
num_rows = math.ceil(len(df_atas_cleaned.columns) / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 4 * num_rows), sharey=True)
axes = axes.flatten()

# Plot frequency counts for each column
for i, col in enumerate(df_atas_cleaned.columns):
    sns.countplot(x=df_atas_cleaned[col], ax=axes[i], palette='viridis', hue=df_atas_cleaned[col], legend=False)
    axes[i].set_title(f'Frequency: {col}')
    axes[i].set_xlabel('Likert Score')
    axes[i].set_ylabel('Count')
    axes[i].set_xticks(range(0, 5))
    axes[i].set_xticklabels(['1', '2', '3', '4', '5'])

# Remove any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

**Descriptive statistics of TIA and altered Hoffman scale for 'Scenario 1'.**

Includes data cleaning, reverse-coding, simple statistics computation, and visualization of data for first evaluation.

---



Trust in Automation (TIA) post-questionnaire after presentation of 'Scenario1' with the XAI navigation assistant.

TIA needs reverse-coding of items 6-11. If not sure whether items need reverse-coding, use correlation plots for negative correlations or try Cronbach's alpha.

In [ ]:
def cronbach_alpha(df):
    k = df.shape[1]
    item_vars = df.var(axis=0, ddof=1).sum()
    total_score_var = df.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - (item_vars / total_score_var))

In [ ]:
# Extract columns containing 'Scenario1_TIA' (change column label if not matching your input)
tia_cols = [col for col in df.columns if 'Scenario1_TIA' in col]
df_tia = df[tia_cols].copy()

# Preprocessing: Convert to numeric and keep only 1-5 range
for col in df_tia.columns:
    df_tia[col] = pd.to_numeric(df_tia[col], errors='coerce')

# Drop rows that are entirely NaN
df_tia_cleaned = df_tia.dropna(how='all')

# Filter for valid Likert range 1-5
df_tia_cleaned = df_tia_cleaned[(df_tia_cleaned >= 1).all(axis=1) & (df_tia_cleaned <= 5).all(axis=1)]
df_tia_cleaned.to_csv('Scenario1_TIA.csv', index=False)

#Rename columns for x-axis readability
plot_df = df_tia_cleaned.copy()
plot_df.columns = [f'TIA_{i+1}' for i in range(len(plot_df.columns))]

df_tia_reversed = plot_df.copy()

# Reverse-code items TIA_6 through TIA_11 (assuming 1-5 scale, 6 - value)
reverse_cols = ['TIA_6', 'TIA_7', 'TIA_8', 'TIA_9', 'TIA_10', 'TIA_11']
for col in reverse_cols:
    df_tia_reversed[col] = 6 - df_tia_reversed[col]

# Calculate Cronbach's alpha on the responses with reverse-coding
cronbach_tia = cronbach_alpha(df_tia_reversed)


In [ ]:
# Calculate statistics for the reverse-coded TIA items and plot results
# Creates bar chart per item with median and std (mean just for reference)
tia_rev_means = df_tia_reversed.mean()
tia_rev_medians = df_tia_reversed.median()
tia_rev_stds = df_tia_reversed.std()

# Visualization
plt.figure(figsize=(14, 6))
sns.barplot(data=df_tia_reversed, palette='viridis', errorbar='sd', capsize=0.1)

# Add median and mean markers
plt.scatter(x=range(len(tia_rev_medians)), y=tia_rev_medians, color='blue', zorder=5, label='Median', marker='o')
plt.scatter(x=range(len(tia_rev_means)), y=tia_rev_means, color='red', marker='*', s=100, zorder=6, label='Mean')

plt.title('Mean and Median Scenario 1 TIA Scores (Items 6-11 Reverse-Coded)')
plt.xlabel('TIA Items')
plt.ylabel('Likert Scale (1-5)')
plt.ylim(1, 5.5)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Display the new statistics table (comment-in if desired)
#tia_rev_stats = df_tia_reversed.describe().T[['mean', '50%', 'std']]
#tia_rev_stats.columns = ['Mean', 'Median', 'Std Dev']
#display(tia_rev_stats)

**EVALUATIONS SCENARIO 1**

**Scenario 1 post-questionnaire based on the Hoffmann scale (Hoffman altered).**

Needs reverse-coding of items 3,4,5,7. (Correlation analysis, Cronbach's alpha)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare the main DataFrame by dropping the first two header/metadata rows
df_cleaned_headers = df.iloc[2:].copy()
# reset the index after iloc
df_cleaned_headers.reset_index(drop=True, inplace=True)

quest_cols = [col for col in df_cleaned_headers.columns if 'Scenario1_Quest' in col]
df_quest = df_cleaned_headers[quest_cols].copy()

for col in df_quest.columns:
    df_quest[col] = pd.to_numeric(df_quest[col], errors='coerce')

# Drop rows where all selected 'Quest' columns are NaN
df_quest_cleaned = df_quest.dropna(how='all')

# Filter to keep only values between 1 and 5 (inclusive)
# This condition needs to be applied to the relevant columns only
if not df_quest_cleaned.empty:
    df_quest_cleaned = df_quest_cleaned[(df_quest_cleaned >= 1).all(axis=1) & (df_quest_cleaned <= 5).all(axis=1)]
df_quest_cleaned = df_quest_cleaned.reset_index(drop=True)

print(f"Cleaned Scenario 1 Quest DataFrame shape: {df_quest_cleaned.shape}")
print("Cleaned Scenario 1 Quest DataFrame head:")
display(df_quest_cleaned.head())

# Check if df_quest_cleaned is empty before proceeding with correlation and plotting.
if df_quest_cleaned.empty:
    print("Warning: df_quest_cleaned is empty after cleaning. Cannot perform correlation analysis or plotting. Please check if 'Scenario1_Quest' columns exist in the raw data.")
else:
    plot_df_quest = df_quest_cleaned.copy()
    plot_df_quest.columns = [f'Quest_{col.split("_")[-1]}' for col in plot_df_quest.columns]

    # Perform correlation analysis to reveal items that might need reverse-coding
    print("\n--- Correlation Analysis for Scenario 1 Quest items (before reverse-coding) ---")
    correlation_matrix_quest = plot_df_quest.corr()
    display(correlation_matrix_quest)

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix_quest, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Matrix of Scenario 1 Quest Items (before reverse-coding)')
    plt.show()

    # Reverse-coding items 3, 4, 5, 7
    df_quest_reversed = plot_df_quest.copy()

    # List of column names to reverse-code based on the item numbers
    reverse_cols_quest_names = [f'Quest_{item_num}' for item_num in [3, 4, 5, 7]]

    print(f"\nItems identified for reverse-coding (based on prior knowledge): {', '.join(reverse_cols_quest_names)}")

    for col in reverse_cols_quest_names:
        if col in df_quest_reversed.columns:
            df_quest_reversed[col] = 6 - df_quest_reversed[col]
            print(f"  - Reverse-coded: {col}")
        else:
            print(f"  - Warning: Column {col} not found for reverse-coding.")

    print("\nFirst 5 rows of DataFrame after reverse-coding:")
    display(df_quest_reversed.head())

    # Calculate Cronbach's alpha on the responses with reverse-coding
    # Check if 'cronbach_alpha' function is defined in a previous code cell or define here adhoc
    try:
        cronbach_quest = cronbach_alpha(df_quest_reversed)
        print(f"\nCronbach's Alpha for Scenario 1 Quest (Hoffmann) after reverse-coding: {cronbach_quest:.3f}")
    except NameError:
        print("\nWarning: 'cronbach_alpha' function not found. Please ensure it's defined in a preceding cell.")


    # Calculate (descriptive) statistics
    quest_rev_means = df_quest_reversed.mean()
    quest_rev_medians = df_quest_reversed.median()
    quest_rev_stds = df_quest_reversed.std()

    print("\nDescriptive Statistics for Scenario 1 Quest (Hoffmann) after reverse-coding:")
    quest_rev_stats = df_quest_reversed.describe().T
    quest_rev_stats['median'] = df_quest_reversed.median()
    quest_rev_stats = quest_rev_stats[['mean', 'median', 'std', 'min', 'max']]
    display(quest_rev_stats)


    # Plot results
    plt.figure(figsize=(14, 6))
    sns.barplot(data=df_quest_reversed, palette='viridis', errorbar='sd', capsize=0.1)

    # Add median and mean markers
    plt.scatter(x=range(len(quest_rev_medians)), y=quest_rev_medians, color='blue', zorder=5, label='Median', marker='o')
    plt.scatter(x=range(len(quest_rev_means)), y=quest_rev_means, color='red', marker='*', s=100, zorder=6, label='Mean')

    plt.title('Corrected Mean and Median Scenario 1 Hoffmann Scores (Items 3,4,5,7 Reverse-Coded) - Altered Hoffman Scale')
    plt.xlabel('Items') # This label will be overwritten by xticks for custom labels
    plt.ylabel('Likert Scale (1-5)')
    plt.ylim(1, 5.5) # Set y-limit to better visualize 1-5 scale and markers
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)

    # Label x-axis as "Item 1", "Item 2", etc.
    num_quest_items = len(df_quest_reversed.columns)
    plt.xticks(ticks=range(num_quest_items), labels=[f'Item {i+1}' for i in range(num_quest_items)])

    plt.tight_layout()
    plt.show()

**Other measures**


---

Item-level and factor analysis on altered Hoffman scale (Scenario 1).


In [ ]:
cronbach_hoffman = cronbach_alpha(df_quest_reversed)
print(f"Cronbach's Alpha for the 8 Hoffman Scale Items Scenario 1 (after reverse-coding): {cronbach_hoffman:.3f}")

In [ ]:
print("Mean Scores of Scenario 1 Hoffman Items (after reverse-coding):")
display(df_quest_reversed.mean().to_frame(name='Mean Score'))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the mean scores (already calculated as quest_rev_means)
mean_scores = df_quest_reversed.mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=mean_scores.index, y=mean_scores.values, palette='viridis', hue=mean_scores.index, legend=False)

plt.title('Mean Scores of Scenario 1 Hoffman Items (Reverse-Coded)')
plt.xlabel('Hoffman Items')
plt.ylabel('Mean Likert Score (1-5)')
plt.ylim(0, 5) # Likert scale is 1-5
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Individual Response Distributions for Hoffman (altered) scale items

In [ ]:
#import pandas as pd
#import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns
import math

print("\n--- Item-level Descriptive Analysis for Scenario 1 Hoffman Scale (df_quest_reversed) ---")

# Calculate comprehensive descriptive statistics for each item (includes count, mean, std, min, 25%, 50% (median), 75%, max)
quest_item_stats = df_quest_reversed.describe().T

# Add skewness and kurtosis for more detailed distribution understanding
quest_item_stats['skewness'] = df_quest_reversed.skew()
quest_item_stats['kurtosis'] = df_quest_reversed.kurt()

print("Descriptive Statistics for each Hoffman Scale Item:")
display(quest_item_stats)

# Plot individual response distributions for each Hoffman item
num_cols = 3 # Number of columns for the subplots
num_items = len(df_quest_reversed.columns)
num_rows = math.ceil(num_items / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 4 * num_rows), sharey=True)
axes = axes.flatten() # Flatten the array of axes for easy iteration

print("\nIndividual Response Distributions for Hoffman Scale Items:")
for i, col in enumerate(df_quest_reversed.columns):
    sns.countplot(x=df_quest_reversed[col], ax=axes[i], palette='viridis', hue=df_quest_reversed[col], legend=False)
    axes[i].set_title(f'Frequency: Item {i+1} - Scenario 1 Hoffman (altered)')
    axes[i].set_xlabel('Likert Score')
    axes[i].set_ylabel('Count')
    # Ensure xticks are set for all possible Likert scores (1-5)
    axes[i].set_xticks(range(1, 6)) # Set ticks from 1 to 5
    axes[i].set_xticklabels(['1', '2', '3', '4', '5'])

# Remove any empty subplots if the number of items is not a perfect multiple of 'num_cols'
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Item-Total Correlations

Item-total correlations measure the correlation of each item with the total score of the scale. Items with low or negative item-total correlations might not be measuring the same construct as the rest of the scale, contributing to low internal consistency (Cronbach's Alpha).

In [ ]:
# Calculate the total score for each respondent across all items
total_score = df_quest_reversed.sum(axis=1)

item_total_correlations = {}

for col in df_quest_reversed.columns:
    score_excluding_item = total_score - df_quest_reversed[col]
    correlation = df_quest_reversed[col].corr(score_excluding_item) # Pearson correlation
    item_total_correlations[col] = correlation

# Conversion pandas Series (for viz.)
item_total_correlations_series = pd.Series(item_total_correlations, name='Item-Total Correlation')

print("Item-Total Correlations for Scenario 1 Hoffman Scale (after reverse-coding):")
display(item_total_correlations_series.sort_values(ascending=False))

**Exploratory Factor Analysis (EFA) for Scenario 1 Hoffman (altered) scale**

---



Given the low Cronbach's alpha and some low/negative item-total correlations, it is possible that the 8 Hoffman scale items are not measuring a single, unidimensional construct. Therefore, we use 'Exploratory Factor Analysis' to identify latent variables.

In [ ]:
!pip install factor_analyzer

import matplotlib.pyplot as plt
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

print("\n--- Exploratory Factor Analysis for Scenario 1 Hoffman Scale ---")

# Check for suitability of Factor Analysis using KMO and Bartlett's Test
# KMO (Kaiser-Meyer-Olkin) Test: measures the sampling adequacy; values > 0.6 are generally considered good
kmo_all_hoff_s1, kmo_model_hoff_s1 = calculate_kmo(df_quest_reversed)
print(f"KMO Test (Scenario 1 Hoffman): {kmo_model_hoff_s1:.3f}")

# Bartlett's Test of Sphericity: tests if the correlation matrix is an identity matrix (i.e., items are unrelated)
# A p-value < 0.05 suggests the data is suitable for factor analysis.
bartlett_p_value_hoff_s1 = calculate_bartlett_sphericity(df_quest_reversed)[1]
print(f"Bartlett's Test p-value (Scenario 1 Hoffman): {bartlett_p_value_hoff_s1:.3f}")

# 2. Determine the optimal number of factors using eigenvalues (Scree Plot)
# First, fit a FactorAnalyzer with the maximum possible number of factors to get all eigenvalues
fa_hoff_s1_scree = FactorAnalyzer(n_factors=df_quest_reversed.shape[1], rotation=None)
fa_hoff_s1_scree.fit(df_quest_reversed)

# Get eigenvalues
ev_hoff_s1, v_hoff_s1 = fa_hoff_s1_scree.get_eigenvalues()
print(f"Eigenvalues (Scenario 1 Hoffman): {ev_hoff_s1}")

# Plot the scree plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(ev_hoff_s1) + 1), ev_hoff_s1, marker='o')
plt.axhline(1, color='red', linestyle='--', label='Kaiser Criterion (Eigenvalue = 1)')
plt.title('Scree Plot for Scenario 1 Hoffman Scale Factor Analysis')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(True)
plt.legend()
plt.show()

# Re-run Factor Analysis with the chosen number of factors using Kaiser Criterion, i.e. keep factors with eigenvalue > 1 (but debated, check for scree plot)
n_factors_hoff_s1_final = sum(ev_hoff_s1 > 1)

# Handle case where no eigenvalue is > 1; default to 1 factor if so.
if n_factors_hoff_s1_final == 0:
    n_factors_hoff_s1_final = 1

print(f"\nUsing {n_factors_hoff_s1_final} factor(s) for Scenario 1 Hoffman Scale Factor Analysis (Kaiser Criterion).")

# Fit the FactorAnalyzer with the chosen number of factors and 'varimax' rotation
fa_hoff_s1_final = FactorAnalyzer(n_factors=n_factors_hoff_s1_final, rotation='varimax')
fa_hoff_s1_final.fit(df_quest_reversed)

print("\nFactor Loadings for Scenario 1 Hoffman Scale:")
hoff_s1_loadings = pd.DataFrame(fa_hoff_s1_final.loadings_, index=df_quest_reversed.columns)
display(hoff_s1_loadings)

print("\nFactor Variance for Scenario 1 Hoffman Scale:")
hoff_s1_factor_variance = pd.DataFrame(fa_hoff_s1_final.get_factor_variance(),
                                     index=['SS Loadings', 'Proportion Var', 'Cumulative Var'],
                                     columns=[f'Factor {i+1}' for i in range(n_factors_hoff_s1_final)])
display(hoff_s1_factor_variance)

**EVALUATIONS SCENARIO 2**

---



**Data extraction for 'Scenario 2' (altered Hoffman, TIA)**

(same data processing and visualizations as for 'Scenario 1'

In [ ]:
scenario2_cols = [col for col in df.columns if 'Scenario2' in col]
scenario2_df = df[scenario2_cols].copy()

print(f"Scenario2 DataFrame shape: {scenario2_df.shape}")
display(scenario2_df.head())

Data cleaning and data extraction.

In [ ]:
# Remove the first two header rows which contain text/metadata
scenario2_df_cleaned = scenario2_df.iloc[2:].copy()

for col in scenario2_df_cleaned.columns:
    scenario2_df_cleaned[col] = pd.to_numeric(scenario2_df_cleaned[col], errors='coerce')

scenario2_df_cleaned = scenario2_df_cleaned.dropna(how='all')

# Filter to keep only values between 1 and 5 (inclusive)
scenario2_df_cleaned = scenario2_df_cleaned[(scenario2_df_cleaned >= 1).all(axis=1) & (scenario2_df_cleaned <= 5).all(axis=1)]
scenario2_df_cleaned = scenario2_df_cleaned.reset_index(drop=True)

# Save the new dataframe to a new csv file
output_file_name_scenario2 = 'Scenario2_cleaned.csv'
scenario2_df_cleaned.to_csv(output_file_name_scenario2, index=False)

print(f"Cleaned Scenario2 DataFrame shape: {scenario2_df_cleaned.shape}")
print("Cleaned Scenario2 DataFrame head:")
display(scenario2_df_cleaned.head())
print(f"Cleaned data saved to '{output_file_name_scenario2}'")

### Reverse-coding for Scenario 2 Hoffman (altered) and TIA items

In [ ]:
# Make a copy of the cleaned data frame for reverse coding
scenario2_df_reversed = scenario2_df_cleaned.copy()

# Identify columns to reverse-code for Hoffman scale (items 3, 4, 5, 7)
hoff_reverse_cols = ['Scenario2_Hoff_3', 'Scenario2_Hoff_4', 'Scenario2_Hoff_5', 'Scenario2_Hoff_7']
for col in hoff_reverse_cols:
    if col in scenario2_df_reversed.columns:
        scenario2_df_reversed[col] = 6 - scenario2_df_reversed[col]

# Identify columns to reverse-code for TIA scale (items 6 through 11)
tia_reverse_cols = ['Scenario2_TIA_6', 'Scenario2_TIA_7', 'Scenario2_TIA_8', 'Scenario2_TIA_9', 'Scenario2_TIA_10', 'Scenario2_TIA_11']
for col in tia_reverse_cols:
    if col in scenario2_df_reversed.columns:
        scenario2_df_reversed[col] = 6 - scenario2_df_reversed[col]

### Cronbach's Alpha for Hoffman (altered) and TIA Scales (Scenario 2) separately

In [ ]:
# Extract Hoff scale columns from the reversed DataFrame
hoff_cols_scenario2 = [col for col in scenario2_df_reversed.columns if 'Scenario2_Hoff' in col]
df_hoff_scenario2_reversed = scenario2_df_reversed[hoff_cols_scenario2]

# Calculate Cronbach's alpha for the Hoff scale
cronbach_hoff_scenario2 = cronbach_alpha(df_hoff_scenario2_reversed)
print(f"Cronbach's Alpha for Scenario 2 Hoff Scale (reverse-coded): {cronbach_hoff_scenario2:.3f}")

# Extract TIA scale columns from the reversed DataFrame
tia_cols_scenario2 = [col for col in scenario2_df_reversed.columns if 'Scenario2_TIA' in col]
df_tia_scenario2_reversed = scenario2_df_reversed[tia_cols_scenario2]

# Calculate Cronbach's alpha for the TIA scale
cronbach_tia_scenario2 = cronbach_alpha(df_tia_scenario2_reversed)
print(f"Cronbach's Alpha for Scenario 2 TIA Scale (reverse-coded): {cronbach_tia_scenario2:.3f}")

### Visualizing Reverse-Coded Hoffman (altered) scale (Scenario 2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate (descriptive) statistics
hoff_rev_means = df_hoff_scenario2_reversed.mean()
hoff_rev_stds = df_hoff_scenario2_reversed.std()
hoff_rev_medians = df_hoff_scenario2_reversed.median()

plt.figure(figsize=(12, 6))
sns.barplot(data=df_hoff_scenario2_reversed, palette='viridis', errorbar='sd', capsize=0.1)

# Add median and mean markers
plt.scatter(x=range(len(hoff_rev_medians)), y=hoff_rev_medians, color='blue', zorder=5, label='Median', marker='o')
plt.scatter(x=range(len(hoff_rev_means)), y=hoff_rev_means, color='red', marker='*', s=100, zorder=6, label='Mean')

plt.title('Mean and Median Scenario 2 Hoffman Scores (Items 3,4,5,7 Reverse-Coded)')
plt.xlabel('Hoffman Items')
plt.ylabel('Likert Scale (1-5)')
plt.ylim(1, 5.5)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Relabel x-axis with 'Item 1', 'Item 2', etc.
num_hoff_items = len(df_hoff_scenario2_reversed.columns)
plt.xticks(ticks=range(num_hoff_items), labels=[f'Item {i+1}' for i in range(num_hoff_items)])

plt.tight_layout()
plt.show()

### Visualizing Reverse-Coded TIA Scale (Scenario 2)

In [ ]:
# Calculate (descriptive) statistics for the reversed TIA scale
tia_rev_means = df_tia_scenario2_reversed.mean()
tia_rev_stds = df_tia_scenario2_reversed.std()
tia_rev_medians = df_tia_scenario2_reversed.median()

plt.figure(figsize=(12, 6))
sns.barplot(data=df_tia_scenario2_reversed, palette='viridis', errorbar='sd', capsize=0.1)

# Add median and mean markers
plt.scatter(x=range(len(tia_rev_medians)), y=tia_rev_medians, color='blue', zorder=5, label='Median', marker='o')
plt.scatter(x=range(len(tia_rev_means)), y=tia_rev_means, color='red', marker='*', s=100, zorder=6, label='Mean')

plt.title('Mean and Median Scenario 2 TIA Scores (Items 6-11 Reverse-Coded)')
plt.xlabel('TIA Items')
plt.ylabel('Likert Scale (1-5)')
plt.ylim(1, 5.5)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Relabel x-axis with 'Item 1', 'Item 2', etc.
num_tia_items = len(df_tia_scenario2_reversed.columns)
plt.xticks(ticks=range(num_tia_items), labels=[f'Item {i+1}' for i in range(num_tia_items)])

plt.tight_layout()
plt.show()

**Correlation between 'Age' and Scenario 2:**

---



To examine the correlation between age and the Hoff and TIA scale responses, we first need to align the 'age' data from the original data frame with the cleaned and reverse-coded Scenario 2 data. Then we can compute and visualize the correlations.

In [ ]:
# Prepare 'age' data aligned with cleaned Scenario 2 responses

# Create a temporary data frame with scenario2 columns and age, retaining original indices
temp_df_with_age = df[scenario2_cols + ['age']].copy()

# Apply initial cleaning steps that were used for `scenario2_df_cleaned`
# Remove the first two header rows (original index 0 and 1)
temp_df_with_age = temp_df_with_age.iloc[2:].copy()

# Convert Scenario2 columns to numeric and 'age' to numeric
for col in scenario2_cols:
    temp_df_with_age[col] = pd.to_numeric(temp_df_with_age[col], errors='coerce')
temp_df_with_age['age'] = pd.to_numeric(temp_df_with_age['age'], errors='coerce')

scenario2_only_for_dropna = temp_df_with_age[scenario2_cols]
all_scenario2_nan_mask = scenario2_only_for_dropna.isnull().all(axis=1)
temp_df_with_age = temp_df_with_age[~all_scenario2_nan_mask].copy()

# Filter to keep only values between 1 and 5 for Scenario2 columns
valid_likert_mask = (temp_df_with_age[scenario2_cols] >= 1).all(axis=1) & \
                    (temp_df_with_age[scenario2_cols] <= 5).all(axis=1)
temp_df_with_age = temp_df_with_age[valid_likert_mask].copy()

# Reverse-codiong for Hoffman scale (items 3, 4, 5, 7)
hoff_reverse_cols_full = [col for col in temp_df_with_age.columns if 'Scenario2_Hoff' in col and col in hoff_reverse_cols]
for col in hoff_reverse_cols_full:
    temp_df_with_age[col] = 6 - temp_df_with_age[col]

# Identify columns to reverse-code for TIA scale (items 6 through 11)
tia_reverse_cols_full = [col for col in temp_df_with_age.columns if 'Scenario2_TIA' in col and col in tia_reverse_cols]
for col in tia_reverse_cols_full:
    temp_df_with_age[col] = 6 - temp_df_with_age[col]

# Drop rows where 'age' is NaN for the final correlation calculation
df_correlation_final = temp_df_with_age.dropna(subset=['age']).copy()

# Ensure 'age' is an integer type (if appropriate)
df_correlation_final['age'] = df_correlation_final['age'].astype(int)

print(f"DataFrame for correlation has {len(df_correlation_final)} rows after cleaning and aligning with 'age'.")
# display(df_correlation_final.head())

#### Calculate Correlations

In [ ]:
# Extract age and item columns for correlation
age_for_corr = df_correlation_final['age']
hoff_items_for_corr = df_correlation_final[[col for col in df_correlation_final.columns if 'Scenario2_Hoff' in col]]
tia_items_for_corr = df_correlation_final[[col for col in df_correlation_final.columns if 'Scenario2_TIA' in col]]

# Calculate correlations
hoff_age_correlations = hoff_items_for_corr.corrwith(age_for_corr)
tia_age_correlations = tia_items_for_corr.corrwith(age_for_corr)

# Rename the index for plotting readability
hoff_age_correlations.index = [f'Hoff Item {i+1}' for i in range(len(hoff_age_correlations))]
tia_age_correlations.index = [f'TIA Item {i+1}' for i in range(len(tia_age_correlations))]

print("Correlation of Age with Hoff Scale Items:")
display(hoff_age_correlations.to_frame(name='Correlation with Age'))

print("\nCorrelation of Age with TIA Scale Items:")
display(tia_age_correlations.to_frame(name='Correlation with Age'))

### Factor Analysis for Hoffman (altered) and TIA Scales (Scenario 2)

In [ ]:
# Install factor_analyzer library, if not already installed
!pip install factor_analyzer

import matplotlib.pyplot as plt
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

# Perform Factor Analysis for Hoffman Scale
print("\n--- Factor Analysis for Hoffman Scale ---")
# Check for suitability of Factor Analysis using KMO and Bartlett's Test
kmo_all, kmo_model = calculate_kmo(df_hoff_scenario2_reversed)
print(f"KMO Test: {kmo_model:.3f}") # A value > 0.6 is generally considered good
bartlett_p_value = calculate_bartlett_sphericity(df_hoff_scenario2_reversed)[1]
print(f"Bartlett's Test p-value: {bartlett_p_value:.3f}") # A p-value < 0.05 suggests data is suitable

# Fitting with all possible factors to determine the optimal number using eigenvalues (scree plot)
fa_hoff = FactorAnalyzer(n_factors=df_hoff_scenario2_reversed.shape[1], rotation=None)
fa_hoff.fit(df_hoff_scenario2_reversed)

# Get eigenvalues
ev, v = fa_hoff.get_eigenvalues()
print(f"Eigenvalues: {ev}")

In [ ]:
# Plot the scree plot to determine the number of factors for the altered Hoffman scale
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(ev) + 1), ev, marker='o')
plt.axhline(1, color='red', linestyle='--', label='Eigenvalue = 1')
plt.title('Scree Plot for Hoff Scale Factor Analysis')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(True)
plt.legend()
plt.show()

# Based on the scree plot, re-run Factor Analysis with the chosen number of factors
n_factors_hoff = sum(ev > 1) # Number of factors with eigenvalue > 1 (Kaiser Criterion)
if n_factors_hoff == 0: # Handle case where no eigenvalue is > 1
    n_factors_hoff = 1

fa_hoff_final = FactorAnalyzer(n_factors=n_factors_hoff, rotation='varimax')
fa_hoff_final.fit(df_hoff_scenario2_reversed)

print(f"\nUsing {n_factors_hoff} factor(s) for Hoff Scale Factor Analysis.")
print("Factor Loadings for Hoff Scale:")
hoff_loadings = pd.DataFrame(fa_hoff_final.loadings_, index=df_hoff_scenario2_reversed.columns)
display(hoff_loadings)

print("Factor Variance for Hoff Scale:")
display(pd.DataFrame(fa_hoff_final.get_factor_variance(), index=['SS Loadings', 'Proportion Var', 'Cumulative Var'], columns=[f'Factor {i+1}' for i in range(n_factors_hoff)]))


In [ ]:
# Perform Factor Analysis for TIA Scale
print("\n--- Factor Analysis for TIA Scale ---")
# Check for suitability of Factor Analysis using KMO and Bartlett's Test
kmo_all_tia, kmo_model_tia = calculate_kmo(df_tia_scenario2_reversed)
print(f"KMO Test: {kmo_model_tia:.3f}") # A value > 0.6 is generally considered good
bartlett_p_value_tia = calculate_bartlett_sphericity(df_tia_scenario2_reversed)[1]
print(f"Bartlett's Test p-value: {bartlett_p_value_tia:.3f}") # A p-value < 0.05 suggests data is suitable

# Perform Factor Analysis with a reasonable number of factors
fa_tia = FactorAnalyzer(n_factors=df_tia_scenario2_reversed.shape[1], rotation=None)
fa_tia.fit(df_tia_scenario2_reversed)

# Get eigenvalues
ev_tia, v_tia = fa_tia.get_eigenvalues()
print(f"Eigenvalues: {ev_tia}")


In [ ]:
# Plot the scree plot to determine the number of factors for TIA scale
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(ev_tia) + 1), ev_tia, marker='o')
plt.axhline(1, color='red', linestyle='--', label='Eigenvalue = 1')
plt.title('Scree Plot for TIA Scale Factor Analysis')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(True)
plt.legend()
plt.show()

# Based on the scree plot, re-run Factor Analysis with the chosen number of factors
n_factors_tia = sum(ev_tia > 1) # Number of factors with eigenvalue > 1 (Kaiser Criterion)
if n_factors_tia == 0: # Handle case where no eigenvalue is > 1
    n_factors_tia = 1

fa_tia_final = FactorAnalyzer(n_factors=n_factors_tia, rotation='varimax')
fa_tia_final.fit(df_tia_scenario2_reversed)

print(f"\nUsing {n_factors_tia} factor(s) for TIA Scale Factor Analysis.")
print("Factor Loadings for TIA Scale:")
tia_loadings = pd.DataFrame(fa_tia_final.loadings_, index=df_tia_scenario2_reversed.columns)
display(tia_loadings)

print("Factor Variance for TIA Scale:")
display(pd.DataFrame(fa_tia_final.get_factor_variance(), index=['SS Loadings', 'Proportion Var', 'Cumulative Var'], columns=[f'Factor {i+1}' for i in range(n_factors_tia)]))


### Paired t-test: Comparing Hoffman Scale Means between Scenario 1 and Scenario 2

Given that the survey likely involves the same participants responding to both scenarios, a paired t-test is appropriate to compare the means of the Hoffman scale between Scenario 1 and Scenario 2.

In [ ]:
from scipy import stats

# Calculate the mean Hoffman score for each participant in Scenario 1
mean_hoff_s1 = df_quest_reversed.mean(axis=1)

# Calculate the mean Hoffman score for each participant in Scenario 2
mean_hoff_s2 = df_hoff_scenario2_reversed.mean(axis=1)

# As per the context, df_quest_reversed (Scenario 1) has 59 rows and df_hoff_scenario2_reversed (Scenario 2) also has 59 rows.
# We assume the indices are aligned due to the sequential cleaning process.

# Perform the paired t-test
t_statistic, p_value = stats.ttest_rel(mean_hoff_s1, mean_hoff_s2)

print(f"Mean Hoffman Score (Scenario 1): {mean_hoff_s1.mean():.3f}")
print(f"Mean Hoffman Score (Scenario 2): {mean_hoff_s2.mean():.3f}")
print(f"\nPaired t-test results for Hoffman Scale (Scenario 1 vs. Scenario 2):\n")
print(f"t-statistic: {t_statistic:.3f}")
print(f"p-value: {p_value:.3f}")

# Set signifance level to 0.05 (significant); change to \alpha=0.01 (very significant) to be more conservative
alpha = 0.05
if p_value < alpha:
    print(f"\nSince the p-value ({p_value:.3f}) is less than the significance level ({alpha}), we reject the null hypothesis.\nThere is a statistically significant difference between the Hoffman scale means in Scenario 1 and Scenario 2.")
else:
    print(f"\nSince the p-value ({p_value:.3f}) is greater than the significance level ({alpha}), we fail to reject the null hypothesis.\nThere is no statistically significant difference between the Hoffman scale means in Scenario 1 and Scenario 2.")

###Paired t-test: Comparing TIA scale means between Scenario 1 and  Scenario 2

In [ ]:
import pandas as pd
from scipy import stats

# Re-define df_tia_reversed for Scenario 1 (to ensure consistent cleaning and availability as apparently some participant answered only in one scenario)
tia_cols_s1 = [col for col in df_cleaned_headers.columns if 'Scenario1_TIA' in col]
df_tia_s1_raw = df_cleaned_headers[tia_cols_s1].copy()

# Preprocessing: Convert to numeric and keep only 1-5 range
for col in df_tia_s1_raw.columns:
    df_tia_s1_raw[col] = pd.to_numeric(df_tia_s1_raw[col], errors='coerce')

# Drop rows that are entirely NaN
df_tia_s1_cleaned = df_tia_s1_raw.dropna(how='all')

# Filter for valid Likert range 1-5
df_tia_s1_cleaned = df_tia_s1_cleaned[(df_tia_s1_cleaned >= 1).all(axis=1) & (df_tia_s1_cleaned <= 5).all(axis=1)]

# Rename columns for consistency (e.g., TIA_1, TIA_2, etc.)
df_tia_s1_renamed = df_tia_s1_cleaned.copy()
df_tia_s1_renamed.columns = [f'TIA_{i+1}' for i in range(len(df_tia_s1_renamed.columns))]

# Reverse-code items TIA_6 through TIA_11
reverse_cols_tia_s1 = ['TIA_6', 'TIA_7', 'TIA_8', 'TIA_9', 'TIA_10', 'TIA_11']
df_tia_s1_reversed = df_tia_s1_renamed.copy()
for col in reverse_cols_tia_s1:
    if col in df_tia_s1_reversed.columns:
        df_tia_s1_reversed[col] = 6 - df_tia_s1_reversed[col]

# Align the two dataframes based on their indices to ensure a paired comparison
# This means keeping only the participants who have valid data in **both** scenarios
common_indices_tia = df_tia_s1_reversed.index.intersection(df_tia_scenario2_reversed.index)

df_tia_s1_aligned = df_tia_s1_reversed.loc[common_indices_tia]
df_tia_s2_aligned = df_tia_scenario2_reversed.loc[common_indices_tia]

# Calculate the mean TIA score for each participant in Scenario 1 from the aligned data
mean_tia_s1 = df_tia_s1_aligned.mean(axis=1)

# Calculate the mean TIA score for each participant in Scenario 2 from the aligned data
mean_tia_s2 = df_tia_s2_aligned.mean(axis=1)

# Perform the paired t-test
t_statistic_tia, p_value_tia = stats.ttest_rel(mean_tia_s1, mean_tia_s2)

print(f"Mean TIA Score (Scenario 1): {mean_tia_s1.mean():.3f}")
print(f"Mean TIA Score (Scenario 2): {mean_tia_s2.mean():.3f}")
print(f"\nPaired t-test results for TIA Scale (Scenario 1 vs. Scenario 2):\n")
print(f"t-statistic: {t_statistic_tia:.3f}")
print(f"p-value: {p_value_tia:.3f}")

# Set significance value alpha=0.05 (significant); change to 0.01 for more conservative measure
alpha = 0.05
if p_value_tia < alpha:
    print(f"\nSince the p-value ({p_value_tia:.3f}) is less than the significance level ({alpha}), we reject the null hypothesis.\nThere is a statistically significant difference between the TIA scale means in Scenario 1 and Scenario 2.")
else:
    print(f"\nSince the p-value ({p_value_tia:.3f}) is greater than the significance level ({alpha}), we fail to reject the null hypothesis.\nThere is no statistically significant difference between the TIA scale means in Scenario 1 and Scenario 2.")